# Phase 2, Task 2 — Clean the Data & Build the Target

You found the mess in Task 1: leading spaces in column names, **1,008 NaN**, **1,586 Inf**, and a multi-class `Label`. A model won't train with NaN/Inf present, so this task is about turning raw, dirty data into a clean numeric `X` and `y` — the CICIDS2017 version of what Task 3 did in Phase 1.

**The plan:**
1. Reload the file.
2. Strip the leading/trailing spaces off every column name (so `' Label'` becomes `'Label'`).
3. Convert `Inf` values into `NaN` (so we can handle them in one shot), then drop the rows with `NaN`.
4. Build a binary target `is_attack` (0 = BENIGN, 1 = any attack).
5. Split into `X` (numeric features) and `y` (`is_attack`).

**Why drop the bad rows instead of fixing them?** We have ~692k rows and only ~2,600 bad ones (<0.4%). Throwing away that tiny fraction is simpler and safer than guessing replacement values. (Later, with rarer attacks, we'd be more careful — but here, dropping is fine.)

## Step 0 — Reload the file
**Your job:** import pandas and numpy, read the Wednesday CSV into `df`.

**Hints:**
- `import pandas as pd` and `import numpy as np`.
- Same `pd.read_csv(...)` path you used in Task 1 (this file has a header — no `header=None`).

In [1]:
# TODO: import pandas & numpy, reload the Wednesday CSV into df
import pandas as pd
df=pd.read_csv("/Users/rohanb/06_projects/CYBERSECURITY/data/raw/CICIDS2017/Wednesday-workingHours.pcap_ISCX.csv")
df

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
692698,53,32215,4,2,112,152,28,28,28.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
692699,53,324,2,2,84,362,42,42,42.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
692700,58030,82,2,1,31,6,31,0,15.500000,21.920310,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
692701,53,1048635,6,2,192,256,32,32,32.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


## Step 1 — Fix the messy column names
Those leading spaces (`' Label'`) are a trap — you'd have to remember the space in every single reference. Let's strip them once, globally.

**Your job:** remove leading/trailing whitespace from every column name.

**Hints:**
- `df.columns = df.columns.str.strip()`  ← strips spaces off all column names at once.
- Confirm it worked: `df.columns.tolist()` should now show `'Label'` (no space), `'Destination Port'`, etc.
- From here on you can write `df['Label']` with no leading space.

In [3]:
# TODO: strip whitespace off all column names, then confirm with df.columns.tolist()
df.columns = df.columns.str.strip()
df.columns.tolist()
df['Label']

0         BENIGN
1         BENIGN
2         BENIGN
3         BENIGN
4         BENIGN
           ...  
692698    BENIGN
692699    BENIGN
692700    BENIGN
692701    BENIGN
692702    BENIGN
Name: Label, Length: 692703, dtype: str

## Step 2 — Handle Inf and NaN
**Term — why Inf exists:** columns like `Flow Bytes/s` are a division (`bytes / duration`). When a flow's duration is ~0, that division blows up to **infinity**. Models can't do math with infinity, so we remove those rows.

**The trick:** first turn every `Inf` into `NaN`, so infinities and missing values become the *same problem* — then drop all `NaN` rows in one step.

**Your job:** replace Inf with NaN, then drop rows containing NaN, then confirm none remain.

**Hints:**
- `df = df.replace([np.inf, -np.inf], np.nan)`  ← both +infinity and -infinity become NaN.
- `df = df.dropna()`  ← drops any row that has a NaN anywhere.
- Check `df.shape` before vs after — you should lose ~2,600 rows (the ~1,008 NaN + ~1,586 Inf you found).
- Final sanity: `df.isna().sum().sum()` should be `0`, and `np.isinf(df.select_dtypes('number')).sum().sum()` should be `0`.

In [6]:
# TODO: Inf -> NaN, drop NaN rows, and confirm zero NaN and zero Inf remain
import numpy as np 
df=df.replace([np.inf, -np.inf], np.nan)
df=df.dropna()
df.shape
df.isna().sum().sum()
np.isinf(df.select_dtypes('number')).sum().sum()

np.int64(0)

## Step 3 — Build the binary target `is_attack`
The `Label` column names the attack type (`BENIGN`, `DoS Hulk`, ...). Like Phase 1, we want a single 0/1 answer: benign vs attack.

**Your job:** make a new column `is_attack` = 0 where `Label == 'BENIGN'`, else 1.

**Hints:**
- Same pattern as Phase 1: `df['is_attack'] = (df['Label'] != 'BENIGN').astype(int)`.
- ⚠️ In this file benign is spelled `'BENIGN'` (all caps) — not `'normal'` like NSL-KDD. Match it exactly.
- Check `df['is_attack'].value_counts()` — should be ~440k zeros and ~252k ones (minus the dropped rows).

In [8]:
# TODO: build the 0/1 is_attack column, then print its value_counts()
df['is_attack']=(df['Label']!="BENIGN").astype(int)
df['is_attack'].value_counts()

is_attack
0    439683
1    251723
Name: count, dtype: int64

## Step 4 — Split into X and y
Same idea as Phase 1: `X` = the clues, `y` = the answer. Here the only columns to drop from `X` are the answer columns.

**Your job:** `y = df['is_attack']`; `X` = everything except the answer columns (`Label` and `is_attack`).

**Hints:**
- `y = df['is_attack']`
- `X = df.drop(columns=['Label', 'is_attack'])`
- (No `difficulty` column here — that was an NSL-KDD thing.) `Label` is the answer in words; `is_attack` is the answer in a number. Both must leave `X`.
- Print `X.shape` and `y.shape`. `X` should have ~78 columns, all numeric, and the same row count as `y`.

In [10]:
# TODO: build y and X, then print both shapes
X=df.drop(columns=['Label','is_attack'])
y=df["is_attack"]
print(X.shape)
print(y.shape)

(691406, 78)
(691406,)


### ✅ You pass Phase 2 Task 2 when:
1. Column names are stripped (no more leading spaces).
2. Zero NaN and zero Inf remain; you can say how many rows you dropped and why.
3. `is_attack` exists as 0/1 (benign vs attack) and you can read its value_counts.
4. `X` (numeric features only, no `Label`/`is_attack`) and `y` exist with matching row counts.

Paste me your before/after shapes, the `is_attack` value_counts, and your final `X.shape` — then Task 3 is training on this data (with a proper train/test split) and seeing whether more attack variety fixes the recall problem.